# Realized Volatility Timing — Rendu projet

## Vue d'ensemble du projet

Ce notebook implémente un système de **timing de volatilité** en deux parties :

### Partie 1 — Backtest de stratégies optionnelles avec delta/gamma hedging
- Génération de positions via `OptionTrade`, `DeltaHedgedOptionTrade`, `DeltaGammaHedgedOptionTrade`
- Backtest P&L décomposé (delta, gamma, theta, vega, résiduel)
- Comparaison avec/sans coûts de transaction (`BacktesterBidAskFromData`, `BacktesterFixedRelativeBidAsk`)

### Partie 2 — Nowcasting de la volatilité réalisée par Filtre de Kalman non-linéaire (UKF)
- Modèle de Heston en espace d'état sur les log-returns
- Unscented Kalman Filter avec correction exacte de la corrélation ρ
- Construction du spread s_t = σ_IV,t − σ̂_t
- Allocation dynamique des stratégies de carry vol selon s_t

### Dynamique de Heston (modèle continu)

$$\begin{cases}
dS_t = \mu S_t \, dt + S_t \sqrt{v_t} \, dW_{1,t} \\
dv_t = \kappa(\theta - v_t) \, dt + \xi \sqrt{v_t} \, dW_{2,t} \\
dW_{1,t} \cdot dW_{2,t} = \rho \, dt
\end{cases}$$

- $v_t$ est la **variance latente** (état caché)
- $S_t$ est le **prix spot observé** (observation)
- L'objectif est d'estimer $\hat{v}_t$ pour en déduire $\hat{\sigma}_t = \sqrt{\hat{v}_t}$
- Puis calculer le spread $s_t = \sigma_{IV,t} - \hat{\sigma}_t$ pour le timing

## 0 — Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import logging
import warnings

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rc("font", **{"size": 16})

In [3]:
# Imports investment_lab
from investment_lab.data.option_db import OptionLoader, SPYOptionLoader, extract_spot_from_options
from investment_lab.data.rates_db import USRatesLoader
from investment_lab.rates import compute_forward
from investment_lab.metrics.volatility import rolling_realized_volatility
from investment_lab.metrics.util import levels_to_returns
from investment_lab.metrics.performance import sharpe_ratio, max_drawdown, calmar_ratio
from investment_lab import option_strategies

# Classes de backtest et de trading
from investment_lab.option_trade import (
    OptionTrade,
    DeltaHedgedOptionTrade,
    DeltaGammaHedgedOptionTrade,
)
from investment_lab.backtest import (
    StrategyBacktester,
    BacktesterBidAskFromData,
    BacktesterFixedRelativeBidAsk,
)

# UKF Heston
from investment_lab.heston_ukf import (
    HestonParams,
    HestonUKF,
    VolatilityTiming,
    build_timing_positions,
)



ImportError: cannot import name 'BacktesterFixedRelativeBidAsk' from 'investment_lab.backtest' (/Users/liessenathan/Desktop/M2/Trading de volatilite/Dauphine-Lecture-Volatility/investment_lab/backtest.py)

In [ ]:
# ── Paramètres globaux ──────────────────────────────────────────────────────
START = datetime(2020, 1, 2)
END   = datetime(2022, 12, 30)
TICKER = "SPY"

print(f"Période : {START.date()} → {END.date()} | Ticker : {TICKER}")

---
## Partie 1 — Backtest de stratégies optionnelles

### 1.1 Génération des trades — Stratégie de base (Short Strangle 95/105, 1W)

On commence par la stratégie de carry vol la plus simple : vente d'un strangle
hebdomadaire 95%/105% (short put K=95%, short call K=105%, maturité 1 semaine).

Le `OptionTrade.generate_trades()` retourne un DataFrame de positions journalières
avec les colonnes : `[date, option_id, entry_date, leg_name, weight, ticker]`.

In [ ]:
# Génération des positions pour le short strangle 95/105 1W
# Rebalancement chaque mercredi (rebal_week_day=[0,2,4] dans la définition de la stratégie)
df_base = OptionTrade.generate_trades(
    start_date=START,
    end_date=END,
    tickers=TICKER,
    legs=option_strategies.SHORT_1W_STRANGLE_95_105,
    cost_neutral=False,
)

print(f"Positions générées : {len(df_base):,} lignes")
print(f"Dates : {df_base['date'].min().date()} → {df_base['date'].max().date()}")
df_base.head(10)

### 1.2 Backtest sans coûts de transaction

Le `StrategyBacktester` décompose le P&L en composantes grecques :
- **delta_pnl** = w × ΔS × δ_{t-1}
- **gamma_pnl** = ½ × w × ΔS² × γ_{t-1}
- **theta_pnl** = w × Δt × θ_{t-1}
- **vega_pnl**  = w × Δσ × ν_{t-1}
- **residual_pnl** = pnl total − somme des grecques

In [ ]:
# Backtest sans coûts de transaction
bt_base = StrategyBacktester(df_base).compute_backtest()

In [ ]:
# ── Métriques de performance ────────────────────────────────────────────────
rets_base = bt_base.nav["NAV"].pct_change().dropna()

print("=== Short Strangle 95/105 1W — Sans coûts de transaction ===")
print(f"  Sharpe Ratio     : {sharpe_ratio(rets_base):.2f}")
print(f"  Max Drawdown     : {max_drawdown(rets_base)*100:.1f}%")
print(f"  Calmar Ratio     : {calmar_ratio(rets_base):.2f}")

In [ ]:
# ── NAV ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))
bt_base.nav["NAV"].plot(ax=ax, label="Short Strangle 95/105 (no tcost)", grid=True)
ax.set_title("NAV — Short Strangle 95/105 1W")
ax.set_ylabel("NAV")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Décomposition P&L cumulé ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 6))
bt_base.pnl.cumsum().plot(ax=ax, grid=True)
ax.set_title("P&L cumulé décomposé — Short Strangle 95/105 1W")
ax.set_ylabel("P&L cumulé")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

### 1.3 Impact des coûts de transaction

On compare trois modèles de coûts :
1. **Sans coût** — mid price utilisé partout
2. **Bid/Ask réel** — bid à l'entrée pour les shorts, ask à la sortie (et inversement pour les longs)
3. **Spread fixe 3%** — demi-spread de 3% du mid appliqué à chaque transaction

In [ ]:
# Backtest avec bid/ask réel
bt_datatcost = BacktesterBidAskFromData(df_base).compute_backtest()

# Backtest avec spread fixe de 3%
bt_fixed3 = BacktesterFixedRelativeBidAsk(df_base).compute_backtest(
    tcost_args={"relative_half_spread": 0.03}
)

In [ ]:
# ── Tableau comparatif ──────────────────────────────────────────────────────
results = []
for name, bt in [
    ("No transaction cost",       bt_base),
    ("Bid-Ask from data",          bt_datatcost),
    ("Fixed 3% half-spread",       bt_fixed3),
]:
    rets = bt.nav["NAV"].pct_change().dropna()
    results.append({
        "Strategy": name,
        "Sharpe":   round(sharpe_ratio(rets), 2),
        "Max DD (%)": round(max_drawdown(rets) * 100, 1),
        "Calmar":   round(calmar_ratio(rets), 2),
    })

pd.DataFrame(results).set_index("Strategy")

In [ ]:
# ── Comparaison NAV ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))
bt_base.nav["NAV"].plot(ax=ax,      label="No transaction cost", grid=True)
bt_datatcost.nav["NAV"].plot(ax=ax, label="Bid-Ask from data")
bt_fixed3.nav["NAV"].plot(ax=ax,    label="Fixed 3% half-spread")
ax.set_title("Impact des coûts de transaction — Short Strangle 95/105 1W")
ax.set_ylabel("NAV")
ax.legend()
plt.tight_layout()
plt.show()

### 1.4 Delta Hedging

Le `DeltaHedgedOptionTrade` ajoute une jambe de couverture delta quotidienne.
Pour chaque groupe (date, ticker, entry_date), il calcule le delta net du portefeuille
et l'annule via une position spot dont l'expiration est fixée au maximum des
expirations des jambes d'options (correction vs l'implémentation originale).

L'effet attendu : réduction de la sensibilité directionnelle, isolement du P&L vega/gamma.

In [ ]:
# Génération des positions avec delta hedging
df_dh = DeltaHedgedOptionTrade.generate_trades(
    start_date=START,
    end_date=END,
    tickers=TICKER,
    legs=option_strategies.SHORT_1W_STRANGLE_95_105,
    cost_neutral=False,
)

print(f"Positions avec delta hedge : {len(df_dh):,} lignes")
print(f"Jambes : {df_dh['leg_name'].unique().tolist()}")

In [ ]:
# Backtest delta-hedgé
bt_dh = StrategyBacktester(df_dh).compute_backtest()

In [ ]:
# ── Vérification : delta portefeuille avant/après hedge ─────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5))

# Delta pondéré sans hedge
df_nd = bt_base.drifted_positions.copy()
df_nd["wgt_delta"] = df_nd["delta"] * df_nd["scaled_weight"]
df_nd.groupby("date")["wgt_delta"].sum().plot(ax=ax1, grid=True,
    title="Delta net du portefeuille — SANS hedge", ylabel="Delta pondéré")

# Delta pondéré avec hedge
df_dh_pos = bt_dh.drifted_positions.copy()
df_dh_pos["wgt_delta"] = df_dh_pos["delta"] * df_dh_pos["scaled_weight"]
df_dh_pos.groupby("date")["wgt_delta"].sum().plot(ax=ax2, grid=True,
    title="Delta net du portefeuille — AVEC delta hedge", ylabel="Delta pondéré")

plt.tight_layout()
plt.show()

In [ ]:
# ── Comparaison P&L décomposé : sans hedge vs delta-hedgé ───────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 6))
bt_base.pnl.cumsum().plot(ax=ax1, title="Sans hedge", grid=True)
bt_dh.pnl.cumsum().plot(ax=ax2, title="Delta hedgé", grid=True)
plt.tight_layout()
plt.show()

### 1.5 Delta-Gamma Hedging

Le `DeltaGammaHedgedOptionTrade` ajoute d'abord une jambe d'option longue
pour couvrir le gamma (réduire la convexité du portefeuille), puis neutralise
le delta résultant via une couverture spot.

In [ ]:
# La jambe de couverture gamma : long put 10D 5 jours
GAMMA_HEDGE_LEG = {
    "day_to_expiry_target": 5,
    "strike_target": -0.1,
    "strike_col": "delta",
    "call_or_put": "P",
    "weight": 0.2,
    "leg_name": "Long 10D Put 5d",
    "rebal_week_day": [2],
}

df_dgh = DeltaGammaHedgedOptionTrade.generate_trades(
    start_date=START,
    end_date=END,
    tickers=TICKER,
    legs=option_strategies.SHORT_1W_STRANGLE_95_105,
    cost_neutral=False,
    hedging_args={"hedging_leg": GAMMA_HEDGE_LEG},
)

print(f"Jambes présentes : {df_dgh['leg_name'].unique().tolist()}")

In [ ]:
# Backtest delta-gamma hedgé
bt_dgh = StrategyBacktester(df_dgh).compute_backtest()

In [ ]:
# ── Tableau de synthèse Partie 1 ────────────────────────────────────────────
results_p1 = []
for name, bt in [
    ("Base (no hedge)",           bt_base),
    ("Delta hedged",               bt_dh),
    ("Delta-Gamma hedged",         bt_dgh),
    ("Base + data tcost",          bt_datatcost),
    ("Base + fixed 3% tcost",      bt_fixed3),
]:
    rets = bt.nav["NAV"].pct_change().dropna()
    results_p1.append({
        "Strategy": name,
        "Sharpe":    round(sharpe_ratio(rets), 2),
        "Max DD (%)": round(max_drawdown(rets) * 100, 1),
        "Calmar":    round(calmar_ratio(rets), 2),
    })

pd.DataFrame(results_p1).set_index("Strategy")

In [ ]:
# ── Comparaison NAV ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 6))
bt_base.nav["NAV"].plot(ax=ax,  label="Base (no hedge)",        grid=True)
bt_dh.nav["NAV"].plot(ax=ax,    label="Delta hedged")
bt_dgh.nav["NAV"].plot(ax=ax,   label="Delta-Gamma hedged")
ax.set_title("Comparaison des stratégies de hedging — Short Strangle 95/105 1W")
ax.set_ylabel("NAV")
ax.legend()
plt.tight_layout()
plt.show()

---
## Partie 2 — Nowcasting de la volatilité réalisée par UKF (dynamique de Heston)

### 2.1 Modèle en espace d'état et discrétisation

Le modèle de Heston est reformulé comme un **State Space Model (SSM)** :

**Équation d'état** (variance latente $v_t$) :
$$v_{t+1} = v_t + \kappa(\theta - v_t)\Delta t + \xi\sqrt{v_t}\sqrt{\Delta t}\,\varepsilon_t^{(2)}$$
$$\text{Bruit de processus : } Q_t = \xi^2 v_t \Delta t$$

**Équation d'observation** (log-return $r_t$ via formule d'Itô) :
$$r_t = \left(\mu - \frac{v_t}{2}\right)\Delta t + \sqrt{v_t \Delta t}\,\varepsilon_t^{(1)}$$
$$\text{Bruit de mesure : } R_t = v_t \Delta t$$

**Terme de corrélation** (spécifique à Heston, absent de filterpy standard) :
$$P_{xz} = \text{Cov}(v_{t+1}, r_t) = \rho \cdot \xi \cdot v_t \cdot \Delta t$$

Ce terme est injecté manuellement dans `HestonUKFCore.update()` avant le calcul
du gain de Kalman $K = P_{xz} \cdot S^{-1}$.

### 2.2 Chargement des données et préparation

In [ ]:
# Chargement des données options et spot SPY
df_options = OptionLoader.load_data(START, END, process_kwargs={"ticker": TICKER})
df_spot = extract_spot_from_options(df_options)
df_rates = USRatesLoader.load_data(START, END)
df_options = compute_forward(df_options=df_options, df_rates=df_rates)

print(f"Options chargées : {len(df_options):,} lignes")
print(f"Dates : {df_options['date'].min().date()} → {df_options['date'].max().date()}")

In [ ]:
# ── Log-returns journaliers depuis le spot ───────────────────────────────────
# levels_to_returns calcule log(S_t / S_{t-1})
log_returns = levels_to_returns(df_spot.set_index("date")["spot"])
log_returns = log_returns.dropna()

print(f"Log-returns : {len(log_returns)} observations")
print(f"Moyenne : {log_returns.mean():.4f} | Std : {log_returns.std():.4f}")

fig, ax = plt.subplots(figsize=(16, 4))
log_returns.plot(ax=ax, grid=True, title="Log-returns journaliers — SPY", alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# ── Volatilité implicite ATM (sigma_IV) ─────────────────────────────────────
# On sélectionne l'option ATM à la monnaie (moneyness ≈ 1) et maturité ~1 mois
from investment_lab.option_selection import select_options

df_atm = select_options(
    df_options,
    call_or_put="C",
    strike_col="moneyness",
    strike_target=1.0,
    day_to_expiry_target=30,
)

# sigma_IV = série temporelle de la vol implicite ATM annualisée
sigma_iv = df_atm.set_index("date")["implied_volatility"]

print(f"sigma_IV : {len(sigma_iv)} dates")
print(f"Moyenne : {sigma_iv.mean():.2%} | Min : {sigma_iv.min():.2%} | Max : {sigma_iv.max():.2%}")

### 2.3 Calibration MLE des paramètres de Heston

On maximise la log-vraisemblance par décomposition en innovations :
$$\log p(r_{1:T} | \kappa, \theta, \xi, \rho, \mu) = -\frac{1}{2} \sum_t \left[ \log(2\pi S_t) + \frac{\nu_t^2}{S_t} \right]$$

où $\nu_t = r_t - \mathbb{E}[r_t | \mathcal{F}_{t-1}]$ est l'innovation du filtre.

L'optimiseur L-BFGS-B est utilisé avec une **pénalité douce sur la condition de Feller** : $2\kappa\theta > \xi^2$ (garantit $v_t > 0$ p.s.).

In [ ]:
# Calibration MLE sur la fenêtre complète (ou sur les 252 derniers jours)
# Cette étape peut prendre ~30-60 secondes selon la machine

ukf_model = HestonUKF(
    initial_params=HestonParams(kappa=2.0, theta=0.04, xi=0.3, rho=-0.7, mu=0.0),
    dt=1.0 / 252.0,
)

ukf_model.fit(log_returns, window=252)

p = ukf_model.params
print("=== Paramètres calibrés ===")
print(f"  κ (kappa) = {p.kappa:.4f}  [vitesse mean-reversion]")
print(f"  θ (theta) = {p.theta:.4f}  [variance long terme → vol {np.sqrt(p.theta):.1%}]")
print(f"  ξ (xi)    = {p.xi:.4f}  [vol-of-vol]")
print(f"  ρ (rho)   = {p.rho:.4f}  [corrélation spot/variance]")
print(f"  μ (mu)    = {p.mu:.6f}  [drift]")
print(f"  Condition de Feller (2κθ > ξ²) : {p.feller_satisfied()}")

### 2.4 Filtrage UKF — estimation de $\hat{v}_t$

Le filtre propage 3 sigma-points (Merwe α=1e-3, β=2, κ=0) à travers la dynamique
de Heston à chaque pas de temps. La correction $\rho \cdot \xi \cdot v_t \cdot \Delta t$
est injectée dans $P_{xz}$ avant le calcul du gain de Kalman.

In [ ]:
# Filtrage sur toute la série
v_hat = ukf_model.filter(log_returns)

# Volatilité estimée annualisée : sigma_hat = sqrt(v_hat)
sigma_hat = ukf_model.sigma_hat

print(f"v_hat estimé — Moyenne : {v_hat.mean():.4f} | Std : {v_hat.std():.4f}")
print(f"sigma_hat estimé — Moyenne : {sigma_hat.mean():.2%}")

In [ ]:
# ── Visualisation : v_hat et sigma_hat vs vol réalisée ───────────────────────
# Volatilité réalisée comme référence
rv_21d = rolling_realized_volatility(log_returns, window=21)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# Variance filtrée
v_hat.plot(ax=ax1, label="v̂_t (UKF Heston)", color="steelblue", grid=True)
ax1.set_title("Variance instantanée estimée v̂_t — UKF Heston")
ax1.set_ylabel("Variance v̂_t")
ax1.legend()

# Volatilité estimée vs vol implicite vs vol réalisée
sigma_hat.plot(ax=ax2, label="σ̂_t = √v̂_t (UKF)", color="steelblue", grid=True)
sigma_iv.reindex(sigma_hat.index).plot(ax=ax2, label="σ_IV,t (ATM 1M)", color="orange", alpha=0.8)
rv_21d.reindex(sigma_hat.index).plot(ax=ax2, label="RV 21j (rolling)", color="gray", alpha=0.6)
ax2.set_title("Comparaison : σ̂_t vs σ_IV vs RV réalisée")
ax2.set_ylabel("Volatilité annualisée")
ax2.legend()

plt.tight_layout()
plt.show()

### 2.5 Spread IV-RV  $s_t = \sigma_{IV,t} - \hat{\sigma}_t$

Le spread mesure la **prime de risque de volatilité** estimée :
- $s_t > 0$ → IV chère par rapport à la RV estimée → carry positif → **short vol profitable**
- $s_t < 0$ → IV bon marché → RV > IV → réduire/fermer la position

In [ ]:
# Calcul du spread s_t = sigma_IV - sigma_hat
spread = ukf_model.implied_realized_spread(sigma_iv)

print(f"Spread s_t — Moyenne : {spread.mean():.2%} | Std : {spread.std():.2%}")
print(f"% jours s_t > 0 (IV chère) : {(spread > 0).mean():.1%}")

fig, ax = plt.subplots(figsize=(16, 5))
spread.plot(ax=ax, label="s_t = σ_IV − σ̂_t", color="steelblue", grid=True)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.fill_between(spread.index, spread, 0,
                where=(spread > 0), alpha=0.2, color="green",  label="IV chère (short vol)")
ax.fill_between(spread.index, spread, 0,
                where=(spread < 0), alpha=0.2, color="red",    label="IV bon marché")
ax.set_title("Spread IV-RV  $s_t = \sigma_{IV,t} - \hat{\sigma}_t$")
ax.set_ylabel("Spread (annualisé)")
ax.legend()
plt.tight_layout()
plt.show()

### 2.6 Signal de timing et allocation dynamique

On normalise $s_t$ en un signal d'allocation $f(s_t) \in [-\text{max\_leverage}, +\text{max\_leverage}]$
puis on l'applique aux poids de la stratégie de base :
$$w_t = w_{\text{base}} \times f(s_t)$$

Trois modes de normalisation sont disponibles :
- **linear** : $f(s_t) = \text{clip}(s_t / \sigma_{s}, \pm L)$ — proportionnel au spread normalisé
- **rank** : percentile roulant recentré — robuste aux outliers  
- **threshold** : signal binaire ±1 au-delà de bandes ±seuil

In [ ]:
# ── Visualisation des 3 modes de signal ─────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

for ax, mode, color in zip(axes, ["linear", "rank", "threshold"], ["steelblue", "darkorange", "seagreen"]):
    timer = VolatilityTiming(scaling=mode, lookback=63, max_leverage=2.0, threshold=0.02)
    sig = timer.compute_signal(spread)
    sig.plot(ax=ax, color=color, grid=True, label=f'Mode "{mode}"')
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_ylabel("Signal")
    ax.legend()

axes[0].set_title("Signaux de timing — 3 modes de normalisation de s_t")
plt.tight_layout()
plt.show()

### 2.7 Backtest avec timing dynamique

On compare le backtest de la stratégie de base avec les versions timées par l'UKF.
On utilise `build_timing_positions()` qui enchaîne les 5 étapes du pipeline.

In [ ]:
# ── Positions avec timing UKF (mode linear) ─────────────────────────────────
df_timed_linear, spread_l, signal_linear = build_timing_positions(
    df_positions   = df_base,
    log_returns    = log_returns,
    sigma_iv       = sigma_iv,
    fit_window     = 252,
    scaling        = "linear",
    lookback       = 63,
    max_leverage   = 2.0,
)

# ── Positions avec timing UKF (mode threshold) ───────────────────────────────
df_timed_thresh, spread_t, signal_thresh = build_timing_positions(
    df_positions   = df_base,
    log_returns    = log_returns,
    sigma_iv       = sigma_iv,
    fit_window     = 252,
    scaling        = "threshold",
    lookback       = 63,
    max_leverage   = 2.0,
    threshold      = 0.02,
)

print("Positions timées générées.")

In [ ]:
# Backtest des stratégies timées
bt_timed_linear = StrategyBacktester(df_timed_linear).compute_backtest()
bt_timed_thresh = StrategyBacktester(df_timed_thresh).compute_backtest()

In [ ]:
# ── Tableau de synthèse complet ──────────────────────────────────────────────
results_final = []
for name, bt in [
    ("Base (no hedge, no timing)",         bt_base),
    ("Delta hedged",                        bt_dh),
    ("UKF Timing — linear (no hedge)",     bt_timed_linear),
    ("UKF Timing — threshold (no hedge)",  bt_timed_thresh),
]:
    rets = bt.nav["NAV"].pct_change().dropna()
    results_final.append({
        "Strategy":    name,
        "Sharpe":      round(sharpe_ratio(rets), 2),
        "Max DD (%)":  round(max_drawdown(rets) * 100, 1),
        "Calmar":      round(calmar_ratio(rets), 2),
    })

pd.DataFrame(results_final).set_index("Strategy")

In [ ]:
# ── Comparaison NAV finale ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 6))
bt_base.nav["NAV"].plot(ax=ax,          label="Base (no timing)",       grid=True, linewidth=1.5)
bt_dh.nav["NAV"].plot(ax=ax,            label="Delta hedged",           linestyle="--")
bt_timed_linear.nav["NAV"].plot(ax=ax,  label="UKF Timing linear",      linewidth=2)
bt_timed_thresh.nav["NAV"].plot(ax=ax,  label="UKF Timing threshold",   linewidth=2)
ax.set_title("Comparaison des stratégies — Short Strangle 95/105 avec UKF Timing")
ax.set_ylabel("NAV")
ax.legend()
plt.tight_layout()
plt.show()

### 2.8 Analyse du signal UKF en période de stress

Le signal $s_t$ devrait baisser (voire devenir négatif) lors des chocs de volatilité
(COVID mars 2020, etc.) car la variance réalisée $\hat{\sigma}_t$ dépasse la vol implicite.
Le filtre de Kalman permet de détecter ce retournement plus rapidement qu'une fenêtre roulante fixe.

In [ ]:
# ── Signal UKF vs distribution des returns ───────────────────────────────────
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

# NAV timée vs base
bt_base.nav["NAV"].plot(ax=ax1, label="Base", grid=True, color="gray", alpha=0.7)
bt_timed_linear.nav["NAV"].plot(ax=ax1, label="UKF Timing", color="steelblue", linewidth=2)
ax1.set_ylabel("NAV")
ax1.legend()
ax1.set_title("NAV — Base vs UKF Timing linear")

# Spread et signal
spread.plot(ax=ax2, color="steelblue", label="s_t (spread IV-RV)", grid=True)
ax2.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax2.set_ylabel("Spread")
ax2.legend()

# Signal normalisé
signal_linear.plot(ax=ax3, color="darkorange", label="Signal linéaire normalisé", grid=True)
ax3.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax3.axhline(1, color="green", linewidth=0.5, linestyle=":")
ax3.axhline(-1, color="red", linewidth=0.5, linestyle=":")
ax3.set_ylabel("Signal f(s_t)")
ax3.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Distribution des retours : base vs timée ────────────────────────────────
rets_base_   = bt_base.nav["NAV"].pct_change().dropna()
rets_timed_  = bt_timed_linear.nav["NAV"].pct_change().dropna()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

rets_base_.hist(bins=80, ax=ax1, alpha=0.7, color="gray",     label="Base")
rets_timed_.hist(bins=80, ax=ax1, alpha=0.7, color="steelblue", label="UKF Timing")
ax1.set_title("Distribution des retours journaliers")
ax1.legend()
ax1.set_xlabel("Retour journalier")

# Scatter : retours base vs signal (vérification que le timing améliore sur les gros moves)
common_idx = rets_base_.index.intersection(signal_linear.index)
ax2.scatter(
    signal_linear.loc[common_idx],
    rets_base_.loc[common_idx],
    alpha=0.2, s=8, color="steelblue"
)
ax2.axhline(0, color="black", linewidth=0.5)
ax2.axvline(0, color="black", linewidth=0.5)
ax2.set_xlabel("Signal f(s_t)")
ax2.set_ylabel("Retour base (lendemain)")
ax2.set_title("Signal UKF vs retour de la stratégie base")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Conclusion

### Partie 1 — Backtest
- Le **short strangle 95/105 1W** capture la prime de risque de volatilité de façon robuste sur 2020-2022
- Le **delta hedging** isole efficacement le P&L vega/gamma et réduit la sensibilité directionnelle
- Les coûts de transaction ont un impact significatif : le spread bid/ask réel dégrade le Sharpe de ~0.3-0.5 points

### Partie 2 — UKF Heston
- Le **filtre de Kalman non-linéaire (UKF)** sur la dynamique de Heston permet d'estimer en temps réel la variance réalisée $\hat{v}_t$ sans utiliser de fenêtre arbitraire
- La **correction de corrélation** $\rho \cdot \xi \cdot v_t \cdot \Delta t$ dans le gain de Kalman est non-négligeable (~57% pour ρ=-0.7, ξ=0.4 typiques des actions)
- Le **spread** $s_t = \sigma_{IV,t} - \hat{\sigma}_t$ constitue un signal de timing efficace : il permet de réduire l'exposition lors des chocs de volatilité (COVID, etc.) et d'augmenter le levier dans les périodes calmes
- Le mode **linear** de normalisation offre un bon compromis entre réactivité et stabilité du signal